In [1]:
import pandas as pd
import numpy as np
import re

from gensim.models import Word2Vec
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

from tensorflow import keras
from tensorflow.keras import layers

In [2]:
data = pd.read_csv("IMDB Dataset.csv")

data.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [3]:
#Preprocess the reviews

stop_words = set(ENGLISH_STOP_WORDS) - {"not", "no", "never"}

def clean_text(text):
    text = re.sub(r'<.*?>', '', str(text))
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    words = text.lower().split()
    words = [word for word in words if word not in stop_words]
    return words

data = data.dropna()

sentences = data["review"].apply(clean_text).tolist()

y = data["sentiment"].map({"positive": 1, "negative": 0}).values

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    sentences, y, test_size=0.2, random_state=42
)

In [5]:
#Train the CBOW Word2Vec model

word2vec_model = Word2Vec(
    X_train,
    vector_size=100,
    window=5,
    min_count=1,
    sg=0,
    epochs=10
)

print("Vocabulary Size:", len(word2vec_model.wv))

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


Vocabulary Size: 186047


In [6]:
#Convert reviews into document vectors

def get_vector(words):
    vectors = [word2vec_model.wv[word] for word in words
               if word in word2vec_model.wv]

    if len(vectors) == 0:
        return np.zeros(100)

    return np.mean(vectors, axis=0)

X_train_vec = np.array([get_vector(words) for words in X_train])
X_test_vec = np.array([get_vector(words) for words in X_test])

print("Training data shape:", X_train_vec.shape)
print("Testing data shape:", X_test_vec.shape)

Training data shape: (40000, 100)
Testing data shape: (10000, 100)


In [7]:
#Define the Feedforward Neural Network

model = keras.Sequential([
    layers.Input(shape=(100,)),
    layers.Dense(64, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense (Dense)                        │ (None, 64)                  │           6,464 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 1)                   │              65 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 6,529 (25.50 KB)

 Trainable params: 6,529 (25.50 KB)

 Non-trainable params: 0 (0.00 B)

In [8]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    X_train_vec, y_train,
    epochs=10,
    batch_size=64,
    validation_split=0.1
)

Epoch 1/10
563/563 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.8346 - loss: 0.3783 - val_accuracy: 0.8618 - val_loss: 0.3295
Epoch 2/10
563/563 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.8590 - loss: 0.3313 - val_accuracy: 0.8650 - val_loss: 0.3193
Epoch 3/10
563/563 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.8597 - loss: 0.3262 - val_accuracy: 0.8692 - val_loss: 0.3165
Epoch 4/10
563/563 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.8623 - loss: 0.3220 - val_accuracy: 0.8650 - val_loss: 0.3210
Epoch 5/10
563/563 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.8640 - loss: 0.3185 - val_accuracy: 0.8670 - val_loss: 0.3141
Epoch 6/10
563/563 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.8650 - loss: 0.3162 - val_accuracy: 0.8705 - val_loss: 0.3116
Epoch 7/10
563/563 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8661 - loss: 0.3125 - val_accuracy: 0.8698 - val_loss: 0.3099
Epoch 8/10
563/563 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.8685 - loss: 0.3100 - val_accuracy: 0.

In [9]:
loss, accuracy = model.evaluate(X_test_vec, y_test)

print("Test Loss:", loss)
print("Test Accuracy:", accuracy * 100, "%")

313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8666 - loss: 0.3151  
Test Loss: 0.3150826692581177
Test Accuracy: 86.6599977016449 %
